In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report


df = pd.read_csv("data/processed_data_last_3.csv", parse_dates=["date"]).copy()

features = [
'diff_elo',
'diff_avg_goals_for', 
'diff_avg_goals_against',
'diff_avg_points', 
'diff_ranking',
'diff_fifa_points',
'ranking_local',
'ranking_away',
'change_local',
'change_away'
]

df_training = df[df["date"] < "2026-05-01"]
df_training = df_training.dropna(subset=features)

df_train = df_training[(df_training["date"] < "2023-06-01") ].copy()
X_train = df_train[features]
y_train = df_train["result"]
df_test = df_training[(df_training["date"] >= "2023-06-01") ].copy()
X_test = df_test[features]
y_test = df_test["result"]
y_test

model = HistGradientBoostingClassifier(
    max_iter=100,  
    learning_rate=0.03,  # pasos cortos
    max_depth=3,  # limitamos la profundidad
    min_samples_leaf=70,  # Exigimos que al menos 30 partidos cumplan una regla para darla por válida
    random_state=45,
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)


precission = accuracy_score(y_test, predictions)

print(f"Precission: {precission * 100:.2f}%")
print("\nFor every result (1=Local, 0=Draw, 2=Away):")
print(classification_report(y_test, predictions))



Precission: 59.95%

For every result (1=Local, 0=Draw, 2=Away):
              precision    recall  f1-score   support

           0       0.33      0.00      0.00       624
           1       0.60      0.89      0.72      1169
           2       0.59      0.64      0.62       759

    accuracy                           0.60      2552
   macro avg       0.51      0.51      0.45      2552
weighted avg       0.53      0.60      0.51      2552



In [11]:

probs = model.predict_proba(X_test)

optimized_predictions = []
treshold = 0.15

for p in probs:
    prob_draw, prob_local, prob_away = p[0], p[1], p[2]
    

    diff_local_away = abs(prob_local - prob_away)
    
    if diff_local_away < treshold:  
        optimized_predictions.append(0) 
    else:
        
        if prob_local > prob_away:
            optimized_predictions.append(1)
        else:
            optimized_predictions.append(2)

optimized_predictions = np.array(optimized_predictions)


print(f"Gradient Boosting : {accuracy_score(y_test, optimized_predictions) * 100:.2f}%")
print("\n Report with Optimized Threshold:")
print(classification_report(y_test, optimized_predictions))

Gradient Boosting : 59.84%

 Report with Optimized Threshold:
              precision    recall  f1-score   support

           0       0.32      0.20      0.25       624
           1       0.64      0.84      0.73      1169
           2       0.66      0.55      0.60       759

    accuracy                           0.60      2552
   macro avg       0.54      0.53      0.53      2552
weighted avg       0.57      0.60      0.57      2552



In [12]:
def optimice_draw(probs, threshold=0.15):

    optimized_predictions = []
    
    for p in probs:
        prob_draw, prob_local, prob_away = p[0], p[1], p[2]
        diff_local_away = abs(prob_local - prob_away)
        
        if diff_local_away < threshold:  
            optimized_predictions.append(0) # Empate por fuerzas igualadas
        else:
            if prob_local > prob_away:
                optimized_predictions.append(1) # Gana Local
            else:
                optimized_predictions.append(2) # Gana Visitante
                
    return np.array(optimized_predictions)

In [13]:
def simulate_match(team_A, team_B, n_steps=10000):
    """
    Busca los datos de dos equipos, aplica simetría y corre 10.000 simulaciones de Montecarlo.
    """

    # Usamos df
    row_A = df[(df['home_team'] == team_A) | (df['away_team'] == team_A)]
    row_B = df[(df['home_team'] == team_B) | (df['away_team'] == team_B)]
    
    if row_A.empty or row_B.empty:
        raise ValueError(f"Make sure tha names of ({team_A} or {team_B}) are in the dataset.")
        
    row_A = row_A.iloc[-1]
    row_B = row_B.iloc[-1]

    # Helper para extraer variables sin importar si jugaron como local o visitante en su último registro
    def extract_metrics(row, team):
        if row['home_team'] == team:
            return row['elo_local'], row['avg_goals_for_local'], row['avg_goals_against_local'], row['avg_points_local'], row['ranking_local'],row['fifa_points_local'], row['change_local']
        else:
            return row['elo_away'], row['avg_goals_for_away'], row['avg_goals_against_away'], row['avg_points_away'], row['ranking_away'], row['fifa_points_away'], row['change_away']

    # Extraemos métricas base individuales
    elo_A, g_f_A, g_a_A, pts_A, r_A, f_pts_A, ch_A = extract_metrics(row_A, team_A)
    elo_B, g_f_B, g_a_B, pts_B, r_B, f_pts_B, ch_B = extract_metrics(row_B, team_B)

    
    data_1 = {
        'diff_elo': elo_A - elo_B, 'diff_avg_goals_for': g_f_A - g_f_B, 'diff_avg_goals_against': g_a_A - g_a_B,
        'diff_avg_points': pts_A - pts_B, 'diff_ranking': r_A - r_B, 'diff_fifa_points': f_pts_A - f_pts_B,
        'ranking_local': r_A, 'ranking_away': r_B, 'change_local': ch_A, 'change_away': ch_B
    }
    # Escenario 2: B es Local, A es Visitante
    data_2 = {
        'diff_elo': elo_B - elo_A, 'diff_avg_goals_for': g_f_B - g_f_A, 'diff_avg_goals_against': g_a_B - g_a_A,
        'diff_avg_points': pts_B - pts_A, 'diff_ranking': r_B - r_A, 'diff_fifa_points': f_pts_B - f_pts_A,
        'ranking_local': r_B, 'ranking_away': r_A, 'change_local': ch_B, 'change_away': ch_A
    }

    # Convertimos a DataFrame respetando el orden exacto de tus features
    df_scenario_1 = pd.DataFrame([data_1])[features]
    df_scenario_2 = pd.DataFrame([data_2])[features]

    # Predicciones probabilísticas del modelo entrenado
    prob_1 = model.predict_proba(df_scenario_1)[0]
    prob_2 = model.predict_proba(df_scenario_2)[0]

    # Promediamos las probabilidades cruzadas para eliminar el sesgo administrativo
    # prob[0]=Empate, prob[1]=Gana Local, prob[2]=Gana Visitante
    p_draw = (prob_1[0] + prob_2[0]) / 2
    p_A = (prob_1[1] + prob_2[2]) / 2
    p_B = (prob_1[2] + prob_2[1]) / 2
    
    total_p = p_draw + p_A + p_B
    p_draw, p_A, p_B = p_draw / total_p, p_A / total_p, p_B / total_p

    resultados_simulados = np.random.choice(
        ['Draw', f'{team_A} Wins', f'{team_B} Wins'], 
        size=n_steps, 
        p=[p_draw, p_A, p_B]
    )

    unique, count = np.unique(resultados_simulados, return_counts=True)
    frequencies = dict(zip(unique, count))

    print(f"MONTECARLO ({n_steps} iterations) | {team_A} vs {team_B}")
    for res, times in frequencies.items():
        percentage = (times / n_steps) * 100
        print(f"{res}: {percentage:.2f}% ({times} times)")
        
    return frequencies

In [14]:
match = simulate_match("Spain", "Uruguay", n_steps=10000)

MONTECARLO (10000 iterations) | Spain vs Uruguay
Draw: 27.39% (2739 times)
Spain Wins: 54.18% (5418 times)
Uruguay Wins: 18.43% (1843 times)
